In [1]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as sklearn

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [4]:
userInputData = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIG = loadDataFromFile("Data:Previous experiments-preprocessed")

C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Loaded 79 records from C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\UserPrevious experiments-preprocessed.json
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Loaded 193493 records from C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\Data:Previous experiments-preprocessed.json


In [5]:
# Convert back to timedelta
timeSeriesData_BIG['timestamp'] = pd.to_timedelta(timeSeriesData_BIG['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIG ["datetime_timestamp"]= timeSeriesData_BIG['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)
columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment', ]
columns_timedelta = ['time taken total',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']

userInputData.loc[:,columns_datetime] = userInputData.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputData.loc[:,columns_timedelta] = userInputData.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [6]:
userInputData["room"].unique()

array(['Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55',
       'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1',
       'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90'],
      dtype=object)

In [8]:
userInputData = userInputData.loc[userInputData["room"]!='Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55']

In [9]:
userInputData["room"].unique()

array(['Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1',
       'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90'],
      dtype=object)

In [10]:
# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}